# 13-phase14-graph-attention

**neuron Phase 14** — attention 의 qkv / out 까지 `HybridGraphLinear` 로 교체 (**block 전체가 graph**). Phase 13 은 FFN 만 graph 였고, Phase 14 에서 paradigm 의 *full graph block* 단계 진입.

핵심 가설:
1. **function preservation (attention 포함)** — full graph block (adj=full/full) ≈ plain block?
2. **attention graph 가 학습에 도움?** — Phase 13 (FFN-only) vs Phase 14 (full graph) 동일 arch 비교에서 final_loss 개선?
3. **dual scale-corrected 우위 재현** — Phase 12/13 의 `hybrid_around_one_around_one` 우위가 full graph 에서도 유지?
4. **파라미터 증가 대비 효과** — full graph 는 attention adj 도 추가되어 파라미터 ↑. ROI?

설계: 4 arch × 2 seed × {Phase 13 (FFN-only), Phase 14 (full graph)} = 16 run.
- arch: plain / hybrid_full_full / hybrid_full_around_one / hybrid_around_one_around_one
- plain 은 use_full_graph 무관 (둘 다 PlainTransformerBlock)
- → 실제 비교: 4 arch × 2 seed × 2 mode - 2 (plain 중복) = 14 unique run

데이터: TinyShakespeare (char-LM, block_size=64)
시드: [42, 123]
작성일: 2026-05-26
연관: Issue [#69](https://github.com/EinSofINTEREST/GraphLM/issues/69) / Phase 13 baseline PR [#68](https://github.com/EinSofINTEREST/GraphLM/pull/68)

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridGraphTransformerLM,
    HybridTransformerTrainConfig,
    count_parameters,
    train_hybrid_transformer_lm,
)
from graphlm.utils import safe_perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

## 1. Config + 데이터

In [ ]:
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

# Phase 13 과 동일 hyperparameter (공정 비교)
HIDDEN_DIM = 128
N_HEADS = 4
FFN_DIM = 256
N_LAYERS = 4
GROUP_SIZE = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500
SEEDS = [42, 123]
ARCHS = [
    "plain",
    "hybrid_full_full",
    "hybrid_full_around_one",
    "hybrid_around_one_around_one",
]
MODES = [("phase13_ffn_only", False), ("phase14_full_graph", True)]

# arch x mode 별 파라미터 수 비교
print("\n== Parameter count by arch × mode ==")
for arch in ARCHS:
    for mode_name, use_full in MODES:
        if arch == "plain" and use_full:
            continue  # plain 은 use_full 무관 (PlainTransformerBlock 동일)
        m = HybridGraphTransformerLM(
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            n_layers=N_LAYERS,
            max_seq_len=BLOCK_SIZE,
            arch=arch,
            group_size=GROUP_SIZE,
            use_full_graph=use_full,
        )
        print(f"  {arch:32s} {mode_name:25s} params = {count_parameters(m):,}")

## 2. Sweep 실행 (4 arch × 2 seed × 2 mode = 14 unique run)

In [ ]:
results = {}
for arch in ARCHS:
    for mode_name, use_full in MODES:
        if arch == "plain" and use_full:
            continue
        for seed in SEEDS:
            key = (arch, mode_name, seed)
            print(f"\n== arch={arch} mode={mode_name} seed={seed} ==")
            cfg = HybridTransformerTrainConfig(
                dataset=dataset,
                vocab_size=vocab_size,
                hidden_dim=HIDDEN_DIM,
                n_heads=N_HEADS,
                ffn_dim=FFN_DIM,
                n_layers=N_LAYERS,
                group_size=GROUP_SIZE,
                arch=arch,
                use_full_graph=use_full,
                block_size=BLOCK_SIZE,
                batch_size=BATCH_SIZE,
                lr=LR,
                max_steps=MAX_STEPS,
                seed=seed,
                device=device,
            )
            out = train_hybrid_transformer_lm(cfg)
            results[key] = out
            print(
                f"  final_loss = {out['final_loss']:.4f}  (perplexity = {safe_perplexity(out['final_loss']):.2f})"
            )

## 3. 결과 표 + 자동 verdict

In [ ]:
print(f"{'arch':32s} {'mode':25s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s}")
print("-" * 95)
for (arch, mode_name, seed), out in results.items():
    fl = out["final_loss"]
    print(f"{arch:32s} {mode_name:25s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f}")

# arch × mode 별 평균
print("\n== Arch × mode summary (mean ± σ across seeds) ==")
summary = {}
for arch in ARCHS:
    for mode_name, use_full in MODES:
        if arch == "plain" and use_full:
            continue
        vals = [results[(arch, mode_name, s)]["final_loss"] for s in SEEDS]
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        summary[(arch, mode_name)] = (mean, std)
        print(
            f"  {arch:32s} {mode_name:25s} {mean:.4f} ± {std:.4f}  (perplexity ≈ {safe_perplexity(mean):.2f})"
        )

# 자동 verdict
print("\n== Verdict ==")
plain_loss = summary[("plain", "phase13_ffn_only")][0]
ff_p14 = summary[("hybrid_full_full", "phase14_full_graph")][0]
aa_p13 = summary[("hybrid_around_one_around_one", "phase13_ffn_only")][0]
aa_p14 = summary[("hybrid_around_one_around_one", "phase14_full_graph")][0]

# 1. function preservation 확장 — full graph 도 plain 근방?
diff_ff_p14 = abs(ff_p14 - plain_loss)
verdict_1 = "PASS" if diff_ff_p14 < 0.15 else "FAIL"
print(
    f"1. full graph function preservation: |hybrid_full_full(p14) - plain| = {diff_ff_p14:.4f}  [{verdict_1}]"
)

# 2. attention graph 효과 — 동일 arch 에서 Phase 14 ≤ Phase 13 + 0.05?
diff_aa = aa_p14 - aa_p13
verdict_2 = "PASS" if diff_aa <= 0.05 else "FAIL"
print(f"2. attention graph not hurting (aa): Phase14 - Phase13 = {diff_aa:+.4f}  [{verdict_2}]")

# 3. 모두 finite
all_finite = all(math.isfinite(out["final_loss"]) for out in results.values())
verdict_3 = "PASS" if all_finite else "FAIL"
print(f"3. all-finite stability (RMSNorm + graph attention): {all_finite}  [{verdict_3}]")

# 4. dual scale-corrected 우위 in full graph — aa_p14 가 ff_p14 보다 ≤?
diff_aa_vs_ff = aa_p14 - ff_p14
verdict_4 = "PASS" if diff_aa_vs_ff <= 0.05 else "FAIL"
print(
    f"4. around_one×around_one ≤ full_full + 0.05 (p14): diff = {diff_aa_vs_ff:+.4f}  [{verdict_4}]"
)

## 4. Loss curve 시각화 — Phase 13 (실선) vs Phase 14 (점선)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = {
    "plain": "tab:gray",
    "hybrid_full_full": "tab:blue",
    "hybrid_full_around_one": "tab:orange",
    "hybrid_around_one_around_one": "tab:green",
}
window = 50

for arch in ARCHS:
    for mode_name, use_full in MODES:
        if arch == "plain" and use_full:
            continue
        losses_per_seed = [results[(arch, mode_name, s)]["losses"] for s in SEEDS]
        # rolling mean — slice 시작점 +1 시프트로 window 와 divisor 일치 (CodeRabbit #3304780219)
        smoothed = []
        for losses in losses_per_seed:
            smoothed.append(
                [
                    sum(losses[max(0, i - window + 1) : i + 1]) / min(i + 1, window)
                    for i in range(len(losses))
                ]
            )
        arr = torch.tensor(smoothed)
        mean = arr.mean(dim=0)
        std = arr.std(dim=0)
        steps = list(range(len(mean)))
        color = colors[arch]
        # Phase 13 = solid, Phase 14 = dashed
        linestyle = "-" if mode_name == "phase13_ffn_only" else "--"
        label = f"{arch} ({mode_name.replace('_', ' ')})"
        ax.plot(steps, mean, label=label, color=color, linewidth=1.5, linestyle=linestyle)
        ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.10)

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title(
    "Phase 14 — full graph block: 4 arch × {Phase 13 FFN-only, Phase 14 full graph} (mean ± σ over 2 seeds)"
)
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase14")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. attention adj 시각화 (Phase 14, hybrid_around_one_around_one 만)

In [ ]:
# Phase 14 의 attention qkv / out 의 학습된 adj_outer 모양 확인 (block 0, seed 42)
snap = results[("hybrid_around_one_around_one", "phase14_full_graph", 42)]["final_adj"][0]
print(f"snap keys (Phase 14 full graph): {sorted(snap.keys())}")

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, layer_name in zip(axes.flat, ["qkv", "out", "fc1", "fc2"], strict=True):
    outer = snap[layer_name]["outer"].numpy()
    im = ax.imshow(outer, cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set_title(f"{layer_name} — adj_outer (block 0)")
    ax.set_xlabel("G_in")
    ax.set_ylabel("G_out")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
fig.savefig(out_dir / "attention_adj.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'attention_adj.png'}")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

- full graph block 도 function preservation 성립?
- attention graph 가 학습에 유의미한 효과?
- qkv vs out 의 adj 학습 패턴 차이?

**Phase 15 후보**:
- sparsity-driven prune — adj magnitude < threshold edge 영구 제거 → dead channel 발생 (DST 계열, training-time dynamic parameter count 의 진정한 첫 단계)
- Net2Net / LiGO 식 grow — 학습 중 channel / group 추가 (function preservation 유지)